In [1]:
import argparse
import pickle

from whoot_model_training.whoot_model_training.models import TimmModel, TimmInputs

from whoot_model_training.whoot_model_training.trainer import WhootTrainer, WhootTrainingArguments
from whoot_model_training.whoot_model_training.data_extractor import buowset_extractor
from whoot_model_training.whoot_model_training.models import TimmModel, TimmInputs
from whoot_model_training.whoot_model_training import CometMLLoggerSupplement

from whoot_model_training.whoot_model_training.preprocessors import MelModelInputPreprocessor

model = TimmModel.from_pretrained("/home/sean/whoot/model_checkpoints/checkpoint-4985", use_safetensors=True)

/home/sean/whoot/.venv/lib/python3.11/site-packages/torch/nn/modules/module.py:2441: UserWarning: for conv_stem.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  warnings.warn(
/home/sean/whoot/.venv/lib/python3.11/site-packages/torch/nn/modules/module.py:2441: UserWarning: for bn1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  warnings.warn(
/home/sean/whoot/.venv/lib/python3.11/site-packages/torch/nn/modules/module.py:2441: UserWarning: for bn1.bias: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model

In [2]:
model

TimmModel(
  (backbone): EfficientNet(
    (conv_stem): Conv2d(1, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn1): BatchNormAct2d(
      32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
      (drop): Identity()
      (act): SiLU(inplace=True)
    )
    (blocks): Sequential(
      (0): Sequential(
        (0): DepthwiseSeparableConv(
          (conv_dw): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (bn1): BatchNormAct2d(
            32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
            (drop): Identity()
            (act): SiLU(inplace=True)
          )
          (aa): Identity()
          (se): SqueezeExcite(
            (conv_reduce): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (act1): SiLU(inplace=True)
            (conv_expand): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (gate): Sigmoid()
          )
          (conv_pw): Con

In [3]:
config = {
    "metadata_csv": "data/burrowing_owl_dataset/metadata.csv",
    "data_path": "data/burrowing_owl_dataset/audio",
    "hf_cache_path": "data/burrowing_owl_dataset/cache/metadata.hf"
}

unlabel_audio_path = "/mnt/restorage/Audiomoth/Raw sound files/2024/RGCB/"
ds = buowset_extractor(
    metadata_csv=config["metadata_csv"],
    parent_path=config["data_path"],
    output_path=config["hf_cache_path"],
)

# ds = buowset_extractor(
#     metadata_csv=config["metadata_csv"],
#     parent_path=config["data_path"],
#     output_path=config["hf_cache_path"],
# )

# Create the model
# model = TimmModel.from_pretrained(model_name)

preprocessor = MelModelInputPreprocessor(TimmInputs, duration=3)

ds["train"].set_transform(preprocessor)

ds

['Unnamed: 0', 'segment', 'labels', 'segment_duration_s', 'fold', 'audio', 'filepath'] ['Unnamed: 0', 'segment', 'labels', 'segment_duration_s', 'fold', 'audio', 'filepath']
cast_storage fake
['audio', 'filepath'] ['audio', 'filepath']
cast_storage fake


Saving the dataset (0/4 shards):   0%|          | 0/15920 [00:00<?, ? examples/s]

embed_storage fake
embed_storage fake
embed_storage fake
embed_storage fake
embed_storage fake
embed_storage fake
embed_storage fake
embed_storage fake
embed_storage fake
embed_storage fake
embed_storage fake
embed_storage fake
embed_storage fake
embed_storage fake
embed_storage fake
embed_storage fake


Saving the dataset (0/2 shards):   0%|          | 0/5081 [00:00<?, ? examples/s]

embed_storage fake
embed_storage fake
embed_storage fake
embed_storage fake
embed_storage fake
embed_storage fake


Saving the dataset (0/2 shards):   0%|          | 0/5042 [00:00<?, ? examples/s]

embed_storage fake
embed_storage fake
embed_storage fake
embed_storage fake
embed_storage fake
embed_storage fake


DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0', 'segment', 'labels', 'segment_duration_s', 'fold', 'audio', 'filepath'],
        num_rows: 15920
    })
    valid: Dataset({
        features: ['Unnamed: 0', 'segment', 'labels', 'segment_duration_s', 'fold', 'audio', 'filepath'],
        num_rows: 5081
    })
    test: Dataset({
        features: ['Unnamed: 0', 'segment', 'labels', 'segment_duration_s', 'fold', 'audio', 'filepath'],
        num_rows: 5042
    })
})

In [4]:
import torch
class RuleBreaker(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x: tuple):
        return self.model.predict(x)
    
model = RuleBreaker(model)

In [5]:
# Modified Demo code from https://github.com/pytorch/executorch/tree/main

from executorch.exir import to_edge_transform_and_lower
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner

# 1. Export your PyTorch model
model = model.eval()
example_inputs = ds["train"][0:5], #the comma is important
example_inputs[0]["spectrogram"] = torch.Tensor(example_inputs[0]["spectrogram"])
example_inputs[0]["labels"] = torch.Tensor(example_inputs[0]["labels"])

exported_program = torch.export.export(model, (example_inputs[0]["spectrogram"],))

# 2. Optimize for target hardware (switch backends with one line)
program = to_edge_transform_and_lower(
    exported_program,
    partitioner=[XnnpackPartitioner()]  # CPU | CoreMLPartitioner() for iOS | QnnPartitioner() for Qualcomm
).to_executorch()

# 3. Save for deployment
with open("model.pte", "wb") as f:
    f.write(program.buffer)

# Test locally via ExecuTorch runtime's pybind API (optional)
from executorch.runtime import Runtime
runtime = Runtime.get()
method = runtime.load_program("model.pte").load_method("forward")
outputs = method.execute((example_inputs[0]["spectrogram"],))

/home/sean/whoot/.venv/lib/python3.11/site-packages/executorch/exir/dialects/edge/_ops.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/home/sean/whoot/.venv/lib/python3.11/site-packages/executorch/exir/dialects/edge/_ops.py:9: UserWarning: Module whoot_model_training was already imported from None, but /home/sean/whoot is being added to sys.path
  import pkg_resources
/tmp/ipykernel_653962/4201338927.py:9: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  example_inputs[0]["spectrogram"] = torch.Tensor(example_inputs[0]["spectrogram"])
[program.cpp:134] 

In [6]:
torch.argmax(outputs[0], dim=1)

tensor([5, 5, 3, 5, 3])

In [7]:
example_inputs[0]["labels"] 

tensor([[0., 0., 0., 1., 0., 0.],
        [0., 0., 0., 1., 0., 0.],
        [0., 0., 0., 1., 0., 0.],
        [0., 0., 0., 1., 0., 0.],
        [0., 0., 0., 0., 1., 0.]])

In [8]:
outputs = method.execute((example_inputs[0]["spectrogram"],))
torch.argmax(outputs[0], dim=1)

tensor([5, 5, 3, 5, 3])

In [ ]:
og_model = TimmModel.from_pretrained("/home/sean/whoot/model_checkpoints/checkpoint-4985", use_safetensors=True)

/home/sean/whoot/.venv/lib/python3.11/site-packages/torch/nn/modules/module.py:2441: UserWarning: for conv_stem.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  warnings.warn(
/home/sean/whoot/.venv/lib/python3.11/site-packages/torch/nn/modules/module.py:2441: UserWarning: for bn1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  warnings.warn(
/home/sean/whoot/.venv/lib/python3.11/site-packages/torch/nn/modules/module.py:2441: UserWarning: for bn1.bias: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model

TypeError: conv2d() received an invalid combination of arguments - got (list, Parameter, NoneType, tuple, tuple, tuple, int), but expected one of:
 * (Tensor input, Tensor weight, Tensor bias = None, tuple of ints stride = 1, tuple of ints padding = 0, tuple of ints dilation = 1, int groups = 1)
      didn't match because some of the arguments have invalid types: (!list of [numpy.ndarray, numpy.ndarray, numpy.ndarray, numpy.ndarray, numpy.ndarray]!, !Parameter!, !NoneType!, !tuple of (int, int)!, !tuple of (int, int)!, !tuple of (int, int)!, !int!)
 * (Tensor input, Tensor weight, Tensor bias = None, tuple of ints stride = 1, str padding = "valid", tuple of ints dilation = 1, int groups = 1)
      didn't match because some of the arguments have invalid types: (!list of [numpy.ndarray, numpy.ndarray, numpy.ndarray, numpy.ndarray, numpy.ndarray]!, !Parameter!, !NoneType!, !tuple of (int, int)!, !tuple of (int, int)!, !tuple of (int, int)!, !int!)


In [13]:
dir(example_inputs[0])

['_MutableMapping__marker',
 '__abstractmethods__',
 '__class__',
 '__class_getitem__',
 '__contains__',
 '__copy__',
 '__delattr__',
 '__delitem__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__ior__',
 '__iter__',
 '__le__',
 '__len__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__or__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__reversed__',
 '__ror__',
 '__setattr__',
 '__setitem__',
 '__sizeof__',
 '__slots__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_abc_impl',
 'clear',
 'copy',
 'data',
 'from_dict',
 'fromkeys',
 'get',
 'items',
 'keys',
 'labels',
 'pop',
 'popitem',
 'setdefault',
 'spectrogram',
 'update',
 'values']

In [16]:
example_inputs[0].spectrogram = torch.Tensor(example_inputs[0].spectrogram )
example_inputs[0].labels = torch.Tensor(example_inputs[0].labels )

In [20]:
og_model(example_inputs[0])["logits"]

tensor([[-13.9122,  -7.6966, -13.5859,  -8.9645, -13.4250,   6.3276],
        [-12.1073,  -6.5152, -12.5275,  -7.6088, -12.3526,   4.4672],
        [-15.9965, -15.6176, -15.6214,   8.4446, -15.8285,  -7.1792],
        [-12.1878, -15.5894, -14.3558,  -3.9112, -13.7771,   3.9622],
        [ -2.0585,  -6.4393,  -8.4049,   0.2084,  -4.9771,  -3.4967]],
       grad_fn=<AddmmBackward0>)